<a href="https://colab.research.google.com/github/shaan-byte/python_ds_colab/blob/main/Data_Joins_and_Merges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Joins & Merges

---

## What we will cover today

1. Why real data lives in multiple tables
2. Inner, Outer, Left, and Right joins — the four core join types
3. Concatenation — stacking DataFrames instead of joining them
4. Handling overlapping column names

---

> **Where we left off:** Over the last three sessions we worked with a single `employees.csv` file — loading it, filtering it, summarising it. But real company data is never in one file. Employee records live in one system, department budgets in another, performance reviews in a third. Today we learn how to combine data from multiple sources — exactly what a database JOIN does, but in Pandas.

---
## Setup — Run this first

In [ ]:
import pandas as pd
import numpy as np
import csv, random

# Recreate the employee dataset — same seed as Sessions 6.1-6.3
random.seed(42)
departments_list = ["Engineering", "Sales", "HR", "Finance"]
cities      = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad"]
first_names = ["Ananya","Rohit","Priya","Vikram","Sneha","Arjun",
               "Deepa","Kiran","Meera","Suresh","Lakshmi","Rahul",
               "Divya","Arun","Pooja","Sanjay","Nisha","Tarun"]
last_names  = ["Sharma","Mehta","Nair","Rao","Pillai","Das",
               "Joshi","Iyer","Bose","Kumar","Reddy","Singh"]
dept_salary = {"Engineering":(70000,150000),"Sales":(55000,110000),
               "HR":(50000,95000),"Finance":(65000,130000)}
rows = []
for i in range(50):
    dept   = random.choice(departments_list)
    lo, hi = dept_salary[dept]
    salary = round(random.uniform(lo, hi), -3)
    perf   = round(random.uniform(2.0, 5.0), 1)
    year   = random.randint(2004, 2023)
    month  = random.randint(1, 12)
    row    = [1001+i,
              random.choice(first_names)+" "+random.choice(last_names),
              dept if random.random()>=0.06 else dept.lower(),
              "" if random.random()<0.08 else salary,
              random.randint(1,20),
              random.choice(cities),
              "" if random.random()<0.08 else perf,
              f"{year}-{month:02d}-01",
              random.choice([True,True,True,False])]
    rows.append(row)
with open("employees.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["employee_id","name","department","salary",
                "experience","city","performance_rating","join_date","active"])
    w.writerows(rows)

emp = pd.read_csv("employees.csv", parse_dates=["join_date"])
emp["salary"]             = pd.to_numeric(emp["salary"],             errors="coerce")
emp["performance_rating"] = pd.to_numeric(emp["performance_rating"], errors="coerce")
emp["department"]         = emp["department"].str.title()

print("employees.csv loaded.")
print(f"Shape: {emp.shape}")
print(emp.head(3))

employees.csv loaded.
Shape: (50, 9)
   employee_id          name   department   salary  experience       city  \
0         1001  Vikram Reddy  Engineering  72000.0          19    Chennai   
1         1002   Tarun Joshi        Sales  68000.0           1      Delhi   
2         1003   Priya Joshi           Hr  57000.0          20  Bangalore   

   performance_rating  join_date  active  
0                 NaN 2011-03-01    True  
1                 3.8 2021-04-01    True  
2                 4.9 2014-02-01   False  


---
# Part 1 — Why Real Data Lives in Multiple Tables

## The problem with one giant table

Imagine the HR system stored every department's budget and head-of-department directly inside every employee row. Every Engineering employee's row would repeat "budget: 5,000,000, head: Vikram Rao" over and over — once per employee. If the budget changes, you would have to update it in dozens of rows instead of one.

This is why real systems split data into separate tables:

```
employees table              departments table
------------------            ------------------
employee_id                   department  (key)
name                          budget
department    <--- shared --->head_of_dept
salary                        location
...
```

The `department` column is the **link** between the two tables. This is called a **key**. To answer a question like "show me each employee along with their department's budget", you need to **join** the two tables together using that shared key.

This is exactly what a SQL `JOIN` does in a database — and `pd.merge()` is Pandas' equivalent.

In [ ]:
# Our second table: department metadata
# Notice: this table does NOT have an entry for every department our employees belong to
# 'Marketing' exists here but has no employees; our employees include all 4 real departments

departments = pd.DataFrame({
    "department":   ["Engineering", "Sales", "Hr", "Marketing"],
    "budget":       [8500000, 4200000, 1800000, 2500000],
    "head_of_dept": ["Vikram Rao", "Priya Nair", "Sneha Pillai", "Arjun Das"],
    "location":     ["Bangalore", "Mumbai", "Delhi", "Mumbai"]
})

print("departments table:")
print(departments)
print()
print("Notice the mismatch: departments has 'Marketing' but no 'Finance'.")
print("Our employees table has 'Finance' but no employee belongs to 'Marketing'.")
print("This mismatch is exactly what makes join types matter.")

departments table:
    department   budget  head_of_dept   location
0  Engineering  8500000    Vikram Rao  Bangalore
1        Sales  4200000    Priya Nair     Mumbai
2           Hr  1800000  Sneha Pillai      Delhi
3    Marketing  2500000     Arjun Das     Mumbai

Notice the mismatch: departments has 'Marketing' but no 'Finance'.
Our employees table has 'Finance' but no employee belongs to 'Marketing'.
This mismatch is exactly what makes join types matter.


In [ ]:
emp

,employee_id,name,department,salary,experience,city,performance_rating,join_date,active
0,1001,Vikram Reddy,Engineering,72000.0,19,Chennai,NaN,2011-03-01,True
1,1002,Tarun Joshi,Sales,68000.0,1,Delhi,3.8,2021-04-01,True
2,1003,Priya Joshi,Hr,57000.0,20,Bangalore,4.9,2014-02-01,False
3,1004,Rahul Kumar,Engineering,NaN,8,Bangalore,3.1,2021-05-01,True
4,1005,Rahul Das,Engineering,100000.0,3,Hyderabad,3.4,2015-03-01,True
5,1006,Lakshmi Sharma,Sales,NaN,11,Chennai,2.8,2021-04-01,True
6,1007,Meera Nair,Hr,60000.0,9,Hyderabad,3.5,2018-03-01,False
7,1008,Priya Sharma,Hr,60000.0,6,Chennai,5.0,2020-08-01,False
8,1009,Ananya Reddy,Finance,104000.0,18,Bangalore,3.4,2012-09-01,True
9,1010,Meera Bose,Engineering,93000.0,4,Bangalore,2.5,2004-12-01,True


---
# Part 2 — Inner, Outer, Left, and Right Joins

## The core idea

`pd.merge()` combines two DataFrames by matching values in a shared column (the **key**). The **join type** determines what happens to rows that do not have a match on the other side.

## Whiteboard: the four join types

```
employees.department:  Engineering, Sales, HR, Finance
departments.department: Engineering, Sales, HR, Marketing

                  Matched          Only in employees    Only in departments
                  (Engineering,                          (Marketing)
                   Sales, HR)       (Finance)

INNER JOIN   ->   keep            drop                  drop
                  (only rows that match on BOTH sides)

LEFT JOIN    ->   keep            keep (NaN for          drop
                                   department info)
                  (everything from the LEFT table, matched info where available)

RIGHT JOIN   ->   keep            drop                  keep (NaN for
                                                          employee info)
                  (everything from the RIGHT table, matched info where available)

OUTER JOIN   ->   keep            keep (NaN)             keep (NaN)
                  (everything from BOTH tables, matched where possible)
```

**Memory aid:** think of LEFT and RIGHT as "which table's rows are guaranteed to survive, no matter what". INNER keeps only certainties on both sides. OUTER keeps everything, gaps and all.

In [ ]:
# INNER JOIN — the default. Keep only rows where the key exists in BOTH tables.

inner = pd.merge(
    emp, departments,
    on  = "department",
    how = "inner"
)

print(f"emp rows:         {len(emp)}")
print(f"departments rows: {len(departments)}")
print(f"inner join rows:  {len(inner)}")
print()

print("Departments present in the result:")
print(inner["department"].unique())
print()
print("Finance employees disappeared (no match in departments table).")
print("Marketing never appeared (no employees in that department).")
print()
print(inner[["name", "department", "salary", "budget", "head_of_dept"]].head(5))

emp rows:         50
departments rows: 4
inner join rows:  38

Departments present in the result:
['Engineering' 'Sales' 'Hr']

Finance employees disappeared (no match in departments table).
Marketing never appeared (no employees in that department).

           name   department    salary   budget  head_of_dept
0  Vikram Reddy  Engineering   72000.0  8500000    Vikram Rao
1   Tarun Joshi        Sales   68000.0  4200000    Priya Nair
2   Priya Joshi           Hr   57000.0  1800000  Sneha Pillai
3   Rahul Kumar  Engineering       NaN  8500000    Vikram Rao
4     Rahul Das  Engineering  100000.0  8500000    Vikram Rao


In [ ]:
# LEFT JOIN — keep ALL rows from the left table (emp), fill with NaN where no match

left = pd.merge(
    emp, departments,
    on  = "department",
    how = "left"
)

print(f"emp rows:        {len(emp)}")
print(f"left join rows:  {len(left)}")
print()
print("Every employee is preserved, even Finance employees (no department match):")
finance_employees = left[left["department"] == "Finance"]
print(finance_employees[["name", "department", "salary", "budget", "head_of_dept"]].head(5))
print()
print("budget and head_of_dept are NaN for Finance — no matching row in departments.")

emp rows:        50
left join rows:  50

Every employee is preserved, even Finance employees (no department match):
            name department    salary  budget head_of_dept
8   Ananya Reddy    Finance  104000.0     NaN          NaN
15    Arun Joshi    Finance   77000.0     NaN          NaN
18    Arun Singh    Finance   82000.0     NaN          NaN
28     Pooja Das    Finance  122000.0     NaN          NaN
29   Nisha Joshi    Finance   86000.0     NaN          NaN

budget and head_of_dept are NaN for Finance — no matching row in departments.


In [ ]:
# RIGHT JOIN — keep ALL rows from the right table (departments), fill with NaN where no match

right = pd.merge(
    emp, departments,
    on  = "department",
    how = "right"
)

print(f"departments rows: {len(departments)}")
print(f"right join rows:  {len(right)}")
print()
print("Marketing appears even though it has no employees:")
marketing_rows = right[right["department"] == "Marketing"]
print(marketing_rows[["name", "department", "salary", "budget", "head_of_dept"]])
print()
print("name and salary are NaN for Marketing — no employee belongs to it.")

departments rows: 4
right join rows:  39

Marketing appears even though it has no employees:
   name department  salary   budget head_of_dept
38  NaN  Marketing     NaN  2500000    Arjun Das

name and salary are NaN for Marketing — no employee belongs to it.


In [ ]:
# OUTER JOIN — keep EVERY row from BOTH tables, NaN wherever there is no match

outer = pd.merge(
    emp, departments,
    on  = "department",
    how = "outer"
)

print(f"emp rows:         {len(emp)}")
print(f"departments rows: {len(departments)}")
print(f"outer join rows:  {len(outer)}")
print()
print("All departments present, including unmatched ones from both sides:")
print(outer["department"].unique())
print()
print("Outer join keeps everything — it is the union of both tables on the key.")

emp rows:         50
departments rows: 4
outer join rows:  51

All departments present, including unmatched ones from both sides:
['Engineering' 'Finance' 'Hr' 'Marketing' 'Sales']

Outer join keeps everything — it is the union of both tables on the key.


In [ ]:
# Side-by-side row count comparison — makes the differences concrete

summary = pd.DataFrame({
    "how":        ["inner", "left", "right", "outer"],
    "row_count":  [len(inner), len(left), len(right), len(outer)],
    "finance_employees_kept": [
        (inner["department"]=="Finance").sum(),
        (left["department"]=="Finance").sum(),
        (right["department"]=="Finance").sum(),
        (outer["department"]=="Finance").sum(),
    ],
    "marketing_dept_kept": [
        (inner["department"]=="Marketing").sum(),
        (left["department"]=="Marketing").sum(),
        (right["department"]=="Marketing").sum(),
        (outer["department"]=="Marketing").sum(),
    ]
})

print(summary)
print()
print("This table is the entire lesson in one glance:")
print("  inner: drops both unmatched groups")
print("  left:  keeps Finance (from emp), drops Marketing")
print("  right: drops Finance, keeps Marketing (from departments)")
print("  outer: keeps both")

     how  row_count  finance_employees_kept  marketing_dept_kept
0  inner         38                       0                    0
1   left         50                      12                    0
2  right         39                       0                    1
3  outer         51                      12                    1

This table is the entire lesson in one glance:
  inner: drops both unmatched groups
  left:  keeps Finance (from emp), drops Marketing
  right: drops Finance, keeps Marketing (from departments)
  outer: keeps both


## Finding out WHERE each row came from

Sometimes you need to know which rows matched and which did not — for example, to find data quality issues. The `indicator=True` argument adds a `_merge` column that tells you exactly that.

In [ ]:
outer_tagged = pd.merge(
    emp, departments,
    on        = "department",
    how       = "outer",
    indicator = True
)

print("The _merge column shows the origin of each row:")
print(outer_tagged["_merge"].value_counts())
print()

print("Rows that only matched on the LEFT (employees with no department record):")
print(outer_tagged[outer_tagged["_merge"] == "left_only"][["name", "department"]])
print()

print("Rows that only matched on the RIGHT (departments with no employees):")
print(outer_tagged[outer_tagged["_merge"] == "right_only"][["department", "budget"]])

The _merge column shows the origin of each row:
_merge
both          38
left_only     12
right_only     1
Name: count, dtype: int64

Rows that only matched on the LEFT (employees with no department record):
             name department
12   Ananya Reddy    Finance
13     Arun Joshi    Finance
14     Arun Singh    Finance
15      Pooja Das    Finance
16    Nisha Joshi    Finance
17     Arun Kumar    Finance
18    Deepa Singh    Finance
19     Tarun Iyer    Finance
20     Divya Iyer    Finance
21  Lakshmi Reddy    Finance
22      Arun Nair    Finance
23    Lakshmi Rao    Finance

Rows that only matched on the RIGHT (departments with no employees):
   department     budget
37  Marketing  2500000.0


In [ ]:
all_data = pd.merge(
    emp, departments,
    on        = "department",
    how       = "outer",
    indicator = True
)

all_data[all_data['_merge'] == 'left_only']

,employee_id,name,department,salary,experience,city,performance_rating,join_date,active,budget,head_of_dept,location,_merge
12,1009.0,Ananya Reddy,Finance,104000.0,18.0,Bangalore,3.4,2012-09-01,True,NaN,NaN,NaN,left_only
13,1016.0,Arun Joshi,Finance,77000.0,4.0,Mumbai,2.3,2017-06-01,True,NaN,NaN,NaN,left_only
14,1019.0,Arun Singh,Finance,82000.0,18.0,Chennai,4.4,2018-05-01,True,NaN,NaN,NaN,left_only
15,1029.0,Pooja Das,Finance,122000.0,8.0,Delhi,NaN,2011-04-01,True,NaN,NaN,NaN,left_only
16,1030.0,Nisha Joshi,Finance,86000.0,11.0,Mumbai,4.6,2012-06-01,True,NaN,NaN,NaN,left_only
17,1034.0,Arun Kumar,Finance,101000.0,15.0,Chennai,2.0,2013-04-01,False,NaN,NaN,NaN,left_only
18,1036.0,Deepa Singh,Finance,105000.0,8.0,Delhi,4.3,2018-07-01,True,NaN,NaN,NaN,left_only
19,1038.0,Tarun Iyer,Finance,105000.0,16.0,Chennai,4.2,2020-07-01,True,NaN,NaN,NaN,left_only
20,1041.0,Divya Iyer,Finance,124000.0,13.0,Chennai,3.8,2004-10-01,True,NaN,NaN,NaN,left_only
21,1042.0,Lakshmi Reddy,Finance,79000.0,6.0,Chennai,3.3,2004-07-01,True,NaN,NaN,NaN,left_only


In [ ]:
pd.merge(emp, departments, on='department', how='inner')

,employee_id,name,department,salary,experience,city,performance_rating,join_date,active,budget,head_of_dept,location
0,1001,Vikram Reddy,Engineering,72000.0,19,Chennai,NaN,2011-03-01,True,8500000,Vikram Rao,Bangalore
1,1002,Tarun Joshi,Sales,68000.0,1,Delhi,3.8,2021-04-01,True,4200000,Priya Nair,Mumbai
2,1003,Priya Joshi,Hr,57000.0,20,Bangalore,4.9,2014-02-01,False,1800000,Sneha Pillai,Delhi
3,1004,Rahul Kumar,Engineering,NaN,8,Bangalore,3.1,2021-05-01,True,8500000,Vikram Rao,Bangalore
4,1005,Rahul Das,Engineering,100000.0,3,Hyderabad,3.4,2015-03-01,True,8500000,Vikram Rao,Bangalore
5,1006,Lakshmi Sharma,Sales,NaN,11,Chennai,2.8,2021-04-01,True,4200000,Priya Nair,Mumbai
6,1007,Meera Nair,Hr,60000.0,9,Hyderabad,3.5,2018-03-01,False,1800000,Sneha Pillai,Delhi
7,1008,Priya Sharma,Hr,60000.0,6,Chennai,5.0,2020-08-01,False,1800000,Sneha Pillai,Delhi
8,1010,Meera Bose,Engineering,93000.0,4,Bangalore,2.5,2004-12-01,True,8500000,Vikram Rao,Bangalore
9,1011,Lakshmi Iyer,Sales,76000.0,10,Delhi,NaN,2020-01-01,True,4200000,Priya Nair,Mumbai


---
### Exercise 1 — Join Types

**Task:** Complete the joins below, then answer the questions using the results.

In [ ]:
# Q1: Perform an inner join between emp and departments on 'department'
q1 = pd.merge(emp, departments, on=___, how=___)
print(f"Q1 Inner join row count: {len(q1)}")

# Q2: Perform a left join — keep all employees, even those without a department match
q2 = pd.merge(emp, departments, on="department", how=___)
print(f"Q2 Left join row count: {len(q2)}")

# Q3: From q2, find employees whose budget is missing (NaN) — these are the unmatched ones
#     Hint: use .isnull() on the budget column
q3 = q2[q2[___].___()]
print(f"\nQ3 Employees with no department budget match: {len(q3)}")
print(q3["department"].unique())

# Q4: Perform an outer join with indicator=True, then count how many rows
#     came from each side using value_counts on the _merge column
q4 = pd.merge(emp, departments, on="department", how="outer", indicator=___)
q4_counts = q4[___].value_counts()
print(f"\nQ4 Merge indicator breakdown:")
print(q4_counts)

print()
print(f"Correct Q1 (< 50 rows, Finance excluded): {len(q1) < 50}")
print(f"Correct Q2 (== 50 rows, all employees kept): {len(q2) == 50}")
print(f"Correct Q3 (Finance employees only): {(q3['department']=='Finance').all()}")

<details>
<summary>Show solution</summary>

```python
q1 = pd.merge(emp, departments, on="department", how="inner")
q2 = pd.merge(emp, departments, on="department", how="left")
q3 = q2[q2["budget"].isnull()]
q4 = pd.merge(emp, departments, on="department", how="outer", indicator=True)
q4_counts = q4["_merge"].value_counts()
```

</details>

---
## A note on join keys: when column names differ

So far our key column was named `department` in both tables, so `on="department"` worked directly. Often, two tables use different names for the same concept — for example, `employee_id` in one table and `staff_id` in another. Use `left_on` and `right_on` instead of `on`.

In [ ]:
# A small table using a differently-named key
parking_assignments = pd.DataFrame({
    "staff_id":     [1001, 1003, 1005, 1099],   # 1099 does not exist in emp
    "parking_spot": ["A-12", "A-15", "B-02", "C-08"]
})

print("parking_assignments table:")
print(parking_assignments)
print()

# left_on / right_on — specify the key column name on each side separately
merged = pd.merge(
    emp, parking_assignments,
    left_on  = "employee_id",
    right_on = "staff_id",
    how      = "left"
)

print("Merged with left_on / right_on (left join):")
print(merged[merged["parking_spot"].notna()][["employee_id", "staff_id", "name", "parking_spot"]])
print()
print("Notice both 'employee_id' AND 'staff_id' appear in the result — they are kept as separate columns.")
print("You can drop the duplicate key afterward: merged.drop(columns='staff_id')")

---
# Part 3 — Concatenation

## Joining vs concatenating — a different problem

Joins combine two tables **side by side**, matching on a key. Concatenation **stacks** tables — either on top of each other (more rows) or side by side (more columns), with no key matching involved.

**When do you concatenate instead of merge?**

The most common case: you have the **same structure** of data from different sources — for example, January's new hires and February's new hires, both with identical columns. You want to combine them into one table, not match them against each other.

```
MERGE  — combine two DIFFERENT tables using a shared key
         (employees + departments, matched on 'department')

CONCAT — stack two tables with the SAME structure
         (January hires + February hires, just appended together)
```

In [ ]:
# pd.concat() stacking rows — the most common use
# Two batches of new hires, same columns, need to become one table

q1_hires = pd.DataFrame({
    "employee_id": [2001, 2002, 2003],
    "name":        ["Farida Khan", "Gaurav Tiwari", "Hema Iyer"],
    "department":  ["Sales", "Engineering", "HR"],
    "salary":      [62000, 88000, 58000]
})

q2_hires = pd.DataFrame({
    "employee_id": [2004, 2005],
    "name":        ["Ibrahim Sheikh", "Jyoti Bhatt"],
    "department":  ["Finance", "Engineering"],
    "salary":      [75000, 91000]
})

print("Q1 hires:")
print(q1_hires)
print()
print("Q2 hires:")
print(q2_hires)
print()

all_hires = pd.concat([q1_hires, q2_hires], ignore_index=True)

print("Combined (stacked) hires:")
print(all_hires)
print()
print(f"Total rows: {len(all_hires)} = {len(q1_hires)} + {len(q2_hires)}")
print()
print("ignore_index=True gives a fresh 0,1,2... index instead of keeping the original 0,1,2 / 0,1 from each piece.")

Q1 hires:
   employee_id           name   department  salary
0         2001    Farida Khan        Sales   62000
1         2002  Gaurav Tiwari  Engineering   88000
2         2003      Hema Iyer           HR   58000

Q2 hires:
   employee_id            name   department  salary
0         2004  Ibrahim Sheikh      Finance   75000
1         2005     Jyoti Bhatt  Engineering   91000

Combined (stacked) hires:
   employee_id            name   department  salary
0         2001     Farida Khan        Sales   62000
1         2002   Gaurav Tiwari  Engineering   88000
2         2003       Hema Iyer           HR   58000
3         2004  Ibrahim Sheikh      Finance   75000
4         2005     Jyoti Bhatt  Engineering   91000

Total rows: 5 = 3 + 2

ignore_index=True gives a fresh 0,1,2... index instead of keeping the original 0,1,2 / 0,1 from each piece.


In [ ]:
# What happens WITHOUT ignore_index — duplicate index values

without_reindex = pd.concat([q1_hires, q2_hires])

print("Without ignore_index=True:")
print(without_reindex)
print()
print("Notice the index: 0, 1, 2, 0, 1 — duplicated!")
print("This can cause confusing bugs later if you use .loc[0] expecting one row.")
print("Always use ignore_index=True when concatenating unless you have a specific reason not to.")

In [ ]:
# Concatenating columns — axis=1
# Useful when two tables have the SAME rows in the SAME order but different columns
# (Less common than row concatenation — usually merge is safer when row order might differ)

contact_info = pd.DataFrame({
    "email": ["farida@company.com", "gaurav@company.com", "hema@company.com"],
    "phone": ["98765-00001", "98765-00002", "98765-00003"]
})

# q1_hires and contact_info have the same number of rows, in matching order
combined_cols = pd.concat([q1_hires, contact_info], axis=1)

print("Concatenated side-by-side (axis=1):")
print(combined_cols)
print()
print("WARNING: axis=1 concat relies entirely on row ORDER, not any key matching.")
print("If the rows were not already aligned, this would silently produce wrong pairings.")
print("When you have a shared identifier, prefer pd.merge() — it is safer.")

Concatenated side-by-side (axis=1):
   employee_id           name   department  salary               email  \
0         2001    Farida Khan        Sales   62000  farida@company.com   
1         2002  Gaurav Tiwari  Engineering   88000  gaurav@company.com   
2         2003      Hema Iyer           HR   58000    hema@company.com   

         phone  
0  98765-00001  
1  98765-00002  
2  98765-00003  

If the rows were not already aligned, this would silently produce wrong pairings.
When you have a shared identifier, prefer pd.merge() — it is safer.


---
### Exercise 2 — Concatenation

**Task:** Three more batches of hires have come in. Combine them correctly.

In [ ]:
q3_hires = pd.DataFrame({
    "employee_id": [2006, 2007],
    "name":        ["Kabir Anand", "Lavanya Menon"],
    "department":  ["Sales", "HR"],
    "salary":      [60000, 56000]
})

q4_hires = pd.DataFrame({
    "employee_id": [2008],
    "name":        ["Mihir Choudhary"],
    "department":  ["Engineering"],
    "salary":      [97000]
})

# Q1: Combine q1_hires, q2_hires, q3_hires, and q4_hires into one DataFrame
#     Use a fresh index
all_year_hires = pd.___([q1_hires, q2_hires, q3_hires, q4_hires], ignore_index=___)
print(f"Q1 Total hires this year: {len(all_year_hires)}")
print(all_year_hires)

# Q2: How many hires were in Engineering across the whole year?
q2_count = (all_year_hires[___] == ___).sum()
print(f"\nQ2 Engineering hires this year: {q2_count}")

# Q3: Append this year's hires onto the MAIN emp DataFrame to get a combined headcount
#     (emp uses 'join_date' and 'active' columns that the hires DataFrames don't have —
#      concat will fill those with NaN automatically for the new rows)
combined_headcount = pd.concat([emp, all_year_hires], ignore_index=___)
print(f"\nQ3 Combined headcount: {len(combined_headcount)} (was {len(emp)} + {len(all_year_hires)} new hires)")

print()
print(f"Correct Q1 (8 total hires): {len(all_year_hires) == 8}")
print(f"Correct Q3 (58 total):      {len(combined_headcount) == 58}")

<details>
<summary>Show solution</summary>

```python
all_year_hires = pd.concat([q1_hires, q2_hires, q3_hires, q4_hires], ignore_index=True)
q2_count = (all_year_hires["department"] == "Engineering").sum()
combined_headcount = pd.concat([emp, all_year_hires], ignore_index=True)
```

</details>

---
# Part 4 — Handling Overlapping Column Names

## The problem

What happens when two tables you are merging have a column with the **same name** that is NOT the join key? Pandas cannot simply combine them — it does not know which one you mean. It needs a way to tell them apart.

In [ ]:
# A ratings table that ALSO has a 'rating' column — same name as something we might expect
# Here, both tables happen to have a column literally called 'rating'

manager_ratings = pd.DataFrame({
    "employee_id": [1001, 1002, 1003, 1004, 1005],
    "rating":      [4.8, 3.9, 4.5, 4.1, 3.7],     # manager's rating
    "reviewed_by": ["Vikram Rao"]*5
})

# Our emp table calls its rating column 'performance_rating' — but let's simulate
# a genuine overlap by renaming temporarily for this demo
emp_demo = emp.rename(columns={"performance_rating": "rating"})

print("emp_demo has a 'rating' column (self-reported / system rating):")
print(emp_demo[["employee_id", "name", "rating"]].head(3))
print()
print("manager_ratings ALSO has a 'rating' column (manager's separate rating):")
print(manager_ratings.head(3))
print()
print("Both tables have a column called exactly 'rating'. What happens when we merge?")

In [ ]:
# Default behaviour: Pandas automatically adds _x and _y suffixes

merged_default = pd.merge(
    emp_demo, manager_ratings,
    on  = "employee_id",
    how = "left"
)

print("Default merge — Pandas auto-resolves the conflict:")
print(merged_default[["employee_id", "name", "rating_x", "rating_y"]].head(5))
print()
print("rating_x came from the LEFT table (emp_demo)")
print("rating_y came from the RIGHT table (manager_ratings)")
print()
print("_x and _y work, but they are not very descriptive. We can do better.")

In [ ]:
# Custom suffixes — far more readable

merged_custom = pd.merge(
    emp_demo, manager_ratings,
    on       = "employee_id",
    how      = "left",
    suffixes = ("_self", "_manager")
)

print("Merge with custom suffixes:")
print(merged_custom[["employee_id", "name", "rating_self", "rating_manager"]].head(5))
print()
print("Much clearer: rating_self vs rating_manager.")
print("Always use descriptive suffixes when you know the columns represent different things.")

In [ ]:
# An alternative strategy: rename BEFORE merging, so there is no conflict at all
# This is often the cleanest approach when you know in advance what the columns mean

manager_ratings_renamed = manager_ratings.rename(columns={"rating": "manager_rating"})

merged_renamed = pd.merge(
    emp_demo.rename(columns={"rating": "self_rating"}),
    manager_ratings_renamed,
    on  = "employee_id",
    how = "left"
)

print("Merge after renaming both sides in advance:")
print(merged_renamed[["employee_id", "name", "self_rating", "manager_rating"]].head(5))
print()
print("No suffixes needed — the column names were made unambiguous beforehand.")
print("This is often the most readable strategy, especially with many overlapping columns.")

### Important distinction: the join key vs other overlapping columns

Notice that `employee_id` — our **join key** — appears only **once** in the result, not as `employee_id_x` / `employee_id_y`. That is because we told `pd.merge()` to use it as the key with `on="employee_id"`. Pandas only adds suffixes to columns that overlap but are **not** part of the key.

If you ever see `_x` / `_y` on a column you expected to be your key, it usually means you used `left_on`/`right_on` with two differently named columns (as we did with `staff_id` earlier) — in that case, both original key columns survive separately, which is correct behaviour, not a bug.

---
### Exercise 3 — Overlapping Columns

**Task:** Two tables below share a column name that is not the join key. Resolve it cleanly.

In [ ]:
# Two snapshots of department info — one from the start of the year, one from now
# Both have a 'budget' column, which will collide on merge

dept_start_of_year = pd.DataFrame({
    "department": ["Engineering", "Sales", "HR", "Finance"],
    "budget":     [8000000, 4000000, 1700000, 3000000]
})

dept_now = pd.DataFrame({
    "department": ["Engineering", "Sales", "HR", "Finance"],
    "budget":     [8500000, 4200000, 1800000, 3100000]
})

# Q1: Merge these two on 'department' using custom suffixes '_start' and '_now'
q1 = pd.merge(
    dept_start_of_year, dept_now,
    on       = ___,
    how      = "inner",
    suffixes = (___, ___)
)
print("Q1 Merged with custom suffixes:")
print(q1)

# Q2: Compute the budget CHANGE for each department (budget_now - budget_start)
q2 = q1[___] - q1[___]
q1["budget_change"] = q2
print("\nQ2 With budget_change column:")
print(q1)

# Q3: Which department had the largest budget increase?
q3_idx = q1["budget_change"].___()
q3_dept = q1.loc[q3_idx, "department"]
print(f"\nQ3 Largest budget increase: {q3_dept}")

print()
print(f"Correct Q1 (columns exist): {'budget_start' in q1.columns and 'budget_now' in q1.columns}")
print(f"Correct Q2 (all positive):  {(q1['budget_change'] > 0).all()}")

<details>
<summary>Show solution</summary>

```python
q1 = pd.merge(
    dept_start_of_year, dept_now,
    on="department", how="inner",
    suffixes=("_start", "_now")
)

q1["budget_change"] = q1["budget_now"] - q1["budget_start"]

q3_idx  = q1["budget_change"].idxmax()
q3_dept = q1.loc[q3_idx, "department"]
```

</details>

---
## Final Exercise — End-to-End Challenge

**Scenario:** The HR analytics team has three separate data sources: the main employee table, a performance review log (multiple reviews per employee, collected over time), and the departments table. You need to produce a single combined report.

This is a realistic multi-table join — exactly the kind of task you would do with real company data.

In [ ]:
# Performance review log — multiple rows per employee, collected quarterly
# Notice: not every employee has a review every quarter, and some employee_ids
# in this log do not exist in our 50-employee table (departed staff, data entry from elsewhere)

random.seed(11)
review_rows = []
quarters = ["2023-Q1", "2023-Q2", "2023-Q3", "2023-Q4"]

for emp_id in emp["employee_id"].sample(35, random_state=11):
    n_reviews = random.randint(1, 3)
    chosen_quarters = random.sample(quarters, n_reviews)
    for q in chosen_quarters:
        review_rows.append({
            "employee_id": emp_id,
            "quarter":     q,
            "review_score": round(random.uniform(2.5, 5.0), 1)
        })

# Add a few reviews for employees NOT in our table (e.g. departed staff)
for fake_id in [9001, 9002, 9003]:
    review_rows.append({
        "employee_id": fake_id,
        "quarter":     random.choice(quarters),
        "review_score": round(random.uniform(2.5, 5.0), 1)
    })

reviews = pd.DataFrame(review_rows)

print(f"reviews table: {len(reviews)} rows (multiple per employee)")
print(reviews.head(8))
print()
print(f"Unique employees with at least one review: {reviews['employee_id'].nunique()}")
print(f"Employees in our main table: {emp['employee_id'].nunique()}")

In [ ]:
# Step 1: Summarise the reviews table down to one row per employee
# (average review_score, count of reviews) — this uses GroupBy from Session 6.3

review_summary = reviews.groupby(___)["review_score"].agg(
    avg_review_score = ___,
    review_count      = ___
).reset_index()

print("Review summary, one row per employee:")
print(review_summary.head(8))
print(f"\nTotal employees with review summaries: {len(review_summary)}")

In [ ]:
# Step 2: Merge the employee table with the review summary
# Use a LEFT join — we want to keep ALL employees, even those with no reviews

with_reviews = pd.merge(
    emp, review_summary,
    on  = ___,
    how = ___
)

print(f"Employees after merging in review data: {len(with_reviews)}  (should still be 50)")
print()

# How many employees have NO reviews at all? (avg_review_score will be NaN)
no_reviews = with_reviews[with_reviews["avg_review_score"].___()]
print(f"Employees with no reviews on record: {len(no_reviews)}")
print(no_reviews[["name", "department"]].head())

In [ ]:
# Step 3: Now merge in the departments table too — building a three-table join
# Use a LEFT join again, joining on 'department'
# Handle the fact that 'budget' doesn't collide here, but department casing/matching might

full_report = pd.merge(
    with_reviews, departments,
    on  = ___,
    how = ___
)

print(f"Final combined table shape: {full_report.shape}")
print()
print(full_report[["name", "department", "salary", "avg_review_score",
                   "review_count", "budget", "head_of_dept"]].head(8))

In [ ]:
# Step 4: Final analysis — find employees who are HIGH performers (review >= 4.3)
# but have NEVER been reviewed more than once (review_count == 1)
# These might be promising employees who need more frequent review cycles

under_reviewed_stars = full_report[
    (full_report["avg_review_score"] ___ 4.3) &
    (full_report["review_count"]      == ___)
][["name", "department", "avg_review_score", "review_count", "salary"]]

print("Under-reviewed high performers:")
print(under_reviewed_stars.sort_values("avg_review_score", ascending=False))

print()
print(f"Correct (full_report still has 50 rows): {len(full_report) == 50}")
print(f"Correct (all results have review_count==1): {(under_reviewed_stars['review_count']==1).all()}")

<details>
<summary>Show solutions for all four steps</summary>

```python
# Step 1
review_summary = reviews.groupby("employee_id")["review_score"].agg(
    avg_review_score = "mean",
    review_count      = "count"
).reset_index()

# Step 2
with_reviews = pd.merge(emp, review_summary, on="employee_id", how="left")
no_reviews   = with_reviews[with_reviews["avg_review_score"].isnull()]

# Step 3
full_report = pd.merge(with_reviews, departments, on="department", how="left")

# Step 4
under_reviewed_stars = full_report[
    (full_report["avg_review_score"] >= 4.3) &
    (full_report["review_count"] == 1)
][["name", "department", "avg_review_score", "review_count", "salary"]]
```

</details>

---
## Session Summary

### Why multiple tables
Real systems split data to avoid repetition and keep each fact in one place. A shared **key** column (like `department` or `employee_id`) links related tables. `pd.merge()` is Pandas' equivalent of a SQL JOIN.

### The four join types

| Join | Keeps | Use when |
|---|---|---|
| `inner` | Only rows matching on BOTH sides | You only want complete, matched records |
| `left` | ALL rows from the left table | You want every record from your "main" table, even unmatched |
| `right` | ALL rows from the right table | You want every record from the "reference" table |
| `outer` | EVERY row from BOTH tables | You want to see all data and all gaps |

- `pd.merge(left_df, right_df, on='key', how='...')` is the core syntax
- `left_on=` / `right_on=` when the key column has different names in each table
- `indicator=True` adds a `_merge` column showing the origin of every row — useful for finding mismatches

### Concatenation
- `pd.concat([df1, df2], ignore_index=True)` stacks DataFrames with the **same structure** — no key matching
- Always use `ignore_index=True` to avoid duplicate index values
- `axis=1` concatenates side by side by row position — risky unless rows are already perfectly aligned; prefer `merge()` when you have a shared identifier
- **Merge** combines different tables via a key. **Concat** stacks tables with matching structure.

### Overlapping column names
- By default, Pandas adds `_x` / `_y` suffixes to non-key columns that exist in both tables
- `suffixes=('_left_label', '_right_label')` gives meaningful names instead
- Renaming columns before merging is often the cleanest strategy when you know the meaning in advance
- The join key itself never gets a suffix — only other overlapping columns do

---

### What comes next

You can now combine data from multiple sources — exactly what real-world analysis requires, since data almost never lives in a single file. The next session moves into **data cleaning**: handling missing values, fixing inconsistent types, and removing duplicates — skills that become especially important once you start merging multiple real-world tables together, since merges are where messy data problems often surface.

---

### Quick reference

```python
import pandas as pd

# Merge (join)
pd.merge(left, right, on='key', how='inner')           # only matches on both sides
pd.merge(left, right, on='key', how='left')             # keep all of left
pd.merge(left, right, on='key', how='right')            # keep all of right
pd.merge(left, right, on='key', how='outer')            # keep everything
pd.merge(left, right, left_on='id', right_on='staff_id') # different key names
pd.merge(left, right, on='key', indicator=True)          # add _merge column
pd.merge(left, right, on='key', suffixes=('_a', '_b'))   # custom overlap suffixes

# Concat (stack)
pd.concat([df1, df2], ignore_index=True)                # stack rows, fresh index
pd.concat([df1, df2], axis=1)                            # stack columns by position

# Finding unmatched rows after an outer join
merged = pd.merge(a, b, on='key', how='outer', indicator=True)
merged[merged['_merge'] == 'left_only']    # only in 'a'
merged[merged['_merge'] == 'right_only']   # only in 'b'
```